In [ ]:
import os
import yaml
import pandas as pd
import numpy as np
from plotnine import *

from tqdm import tqdm
from sklearn.metrics import r2_score


In [ ]:
pdir = '/s/project/geno2pheno/funcrvp/paper_revisions/supplementary_results/power_analysis'
base_dir = '/s/project/geno2pheno/funcrvp/paper_revisions'
lm_cov = f"{base_dir}/predictions/all_traits_covariates_only_phenopred_filteredv3.pq"

rvat_path = f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat_sampling"
funcrvp_path = f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/pops_mat_pca256_omics/funcrvp_better_filteredv3_sampling"


## FuncRVP results

In [ ]:
beta_df_list = []
pred_df_list = []

for s in tqdm([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]):
    # Save the concatenated dataframes to parquet files
    temp_b = pd.read_parquet(os.path.join(funcrvp_path+str(s), "all_traits_betas.pq"))
    temp_b['sampling'] = s
    beta_df_list.append(temp_b)
    temp_p = pd.read_parquet(os.path.join(funcrvp_path+str(s), "all_traits_phenopred.pq"))
    temp_p['sampling'] = s
    pred_df_list.append(temp_p)

# Add the actual model (sampling = 1)
temp_b = pd.read_parquet(f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/pops_mat_pca256_omics/funcrvp_better_filteredv3/all_traits_betas.pq")
temp_b['sampling'] = 1
beta_df_list.append(temp_b)

temp_p = pd.read_parquet(f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/pops_mat_pca256_omics/funcrvp_better_filteredv3/all_traits_phenopred.pq")
temp_p['sampling'] = 1
pred_df_list.append(temp_p)

pd.concat(beta_df_list).to_parquet(f'{pdir}/funcrvp_betas_all_sampling.pq')
pd.concat(pred_df_list).to_parquet(f'{pdir}/funcrvp_phenopred_all_sampling.pq')


## RVAT results

In [ ]:
beta_df_list = []
for s in tqdm([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]):
    temp_b = pd.read_parquet(os.path.join(rvat_path+str(s), "all_traits_rvat.pq"))
    temp_b['sampling'] = s
    beta_df_list.append(temp_b)

# Add the actual model (sampling = 1)
temp_b = pd.read_parquet(f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_rvat.pq")
temp_b['sampling'] = 1
beta_df_list.append(temp_b)

pd.concat(beta_df_list).to_parquet(f'{pdir}/rvat_betas_all_sampling.pq')

In [ ]:
p_thresh_list = [0.05]
for p_thresh in p_thresh_list:
    pred_df_list = []
    for s in tqdm([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]):
        temp_p = pd.read_parquet(os.path.join(rvat_path+str(s), f"all_traits_phenopred_{p_thresh}.pq")).rename(columns={'pred': 'best_prediction'})
        temp_p['sampling'] = s
        pred_df_list.append(temp_p)

    temp_p = pd.read_parquet(f"{base_dir}/predictions/ukbb_wes_500k_DeepRVAT_final_090924_medianshifted/rvat/all_traits_phenopred_{p_thresh}.pq")
    temp_p['sampling'] = 1
    pred_df_list.append(temp_p)
    pd.concat(pred_df_list).to_parquet(f'{pdir}/rvat_phenopred_{p_thresh}_all_sampling.pq')


## LM - covariates only results

In [ ]:
temp_p = pd.read_parquet(lm_cov).rename(columns={'best_prediction': 'cov_prediction'})
temp_p.individual

In [ ]:
p_thresh_list = [0.05]
for p_thresh in p_thresh_list:
    pred_df_list = []
    for s in tqdm([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]):
        temp_p = pd.read_parquet(os.path.join(rvat_path+str(s)+'_onlycov', f"all_traits_phenopred_{p_thresh}.pq")).rename(columns={'pred': 'cov_prediction'})
        temp_p['individual'] = temp_p['individual'].astype(str)
        temp_p['sampling'] = s
        pred_df_list.append(temp_p)

    temp_p = pd.read_parquet(lm_cov).rename(columns={'best_prediction': 'cov_prediction'})
    temp_p['sampling'] = 1
    temp_p['only_covariates'] = True
    pred_df_list.append(temp_p)
    pd.concat(pred_df_list).to_parquet(f'{pdir}/lm_cov_phenopred_all_sampling.pq')

## gene-trait associations

In [ ]:
rb = pd.read_parquet(f'{pdir}/rvat_betas_all_sampling.pq')
rb['rvat_significant'] = rb['pval'] < 0.05/rb.gene_id.nunique()
rb

In [ ]:
srb = rb.groupby(['sampling', 'trait'])['rvat_significant'].sum().reset_index()
srb

In [ ]:
srb1 = srb[srb.sampling==1][['trait', 'rvat_significant']].rename(columns={'rvat_significant': 'total_significant'})
srb1

In [ ]:
srb = srb.merge(srb1, on='trait')
srb['signif_frac'] = srb['rvat_significant']/srb['total_significant']
srb

## Phenotype prediction

In [ ]:
cov = pd.read_parquet(f'{pdir}/lm_cov_phenopred_all_sampling.pq')
cov

In [ ]:
cov_r2 = pd.DataFrame(cov.groupby(['sampling', 'trait']).apply(lambda group: r2_score(group['trait_measurement'], group['cov_prediction']))).reset_index()
cov_r2.columns = ['sampling', 'trait', 'cov_r2']
cov_r2

In [ ]:
fp = pd.read_parquet(f'{pdir}/funcrvp_phenopred_all_sampling.pq')
fp

In [ ]:
fp_r2 = pd.DataFrame(fp.groupby(['sampling', 'trait']).apply(lambda group: r2_score(group['trait_measurement'], group['best_prediction']))).reset_index()
fp_r2.columns = ['sampling', 'trait', 'funcrvp_r2']
fp_r2

In [ ]:
p_thresh = 0.05
rp = pd.read_parquet(f'{pdir}/rvat_phenopred_{p_thresh}_all_sampling.pq')
rp

In [ ]:
rp_r2 = pd.DataFrame(rp.groupby(['sampling', 'trait']).apply(lambda group: r2_score(group['trait_measurement'], group['best_prediction']))).reset_index()
rp_r2.columns = ['sampling', 'trait', 'lm_r2']
rp_r2

In [ ]:
rb = pd.read_parquet(f'{pdir}/rvat_betas_all_sampling.pq')
rb['rvat_significant'] = rb['pval'] < 0.05/rb.gene_id.nunique()
srb = rb.groupby(['sampling', 'trait'])['rvat_significant'].sum().reset_index()
srb

In [ ]:
plot_df = srb.merge(cov_r2, on=['sampling', 'trait'], how='outer').merge(fp_r2, on=['sampling', 'trait'], how='outer').merge(rp_r2, on=['sampling', 'trait'], how='outer')

# plot_df = srb.merge(cov_r2, on=['sampling', 'trait'], how='outer').merge(fp_r2, on=['sampling', 'trait'], how='outer').merge(rp_r2, on=['sampling', 'trait'], how='outer')

plot_df

In [ ]:
plot_df['funcrvp_rel_del_r2'] = (plot_df['funcrvp_r2'] - plot_df['cov_r2'])/plot_df['cov_r2']
plot_df['lm_rel_del_r2'] = (plot_df['lm_r2'] - plot_df['cov_r2'])/plot_df['cov_r2']
plot_df['funcrvp_lm_diff'] = plot_df['funcrvp_r2'] - plot_df['lm_r2']
plot_df['rel_del_diff'] = (plot_df['funcrvp_r2'] - plot_df['lm_r2'])/plot_df['lm_r2']
plot_df['improved'] = plot_df['funcrvp_rel_del_r2'] > plot_df['lm_rel_del_r2']
plot_df

In [ ]:
plot_df['always_improved'] = plot_df.groupby(['trait'])['funcrvp_rel_del_r2'].transform(lambda x: (x > 0).all())
plot_df['always_beat_lm'] = plot_df.groupby(['trait'])['funcrvp_lm_diff'].transform(lambda x: (x > 0).all())
plot_df

In [ ]:
(
    ggplot(plot_df, aes(y='funcrvp_rel_del_r2', x='sampling', color='always_beat_lm')) +
    geom_line(aes(fill='trait')) +
    geom_point() +
    xlab('train-val sampling') +
    ylab('FuncRVP relative delta R^2') +
    theme_bw() +
    theme(
        figure_size=(8, 5),
        legend_position='right'
    )
)

In [ ]:
from statsmodels.formula.api import ols

# Fit a linear model
model = ols('funcrvp_rel_del_r2 ~ sampling', data=plot_df).fit()

(
    ggplot(plot_df, aes(y='funcrvp_rel_del_r2', x='sampling')) +
    geom_point() +
    stat_smooth(method='lm') +
    xlab('Proportion of individuals in training+validation set') +
    ylab('FuncRVP relative delta R^2') +
    annotate("text", x=0.5, y=max(plot_df['funcrvp_rel_del_r2']), 
        label=f"Slope: {model.params['sampling']:.4f}\nP-value: {model.pvalues['sampling']:.4e}", 
        ha='center', va='top', size=12, color='red') +
    theme_bw() +
    theme(
        text=element_text(size=12)
    )
)

In [ ]:
from statsmodels.formula.api import ols

# Fit a linear model
model = ols('funcrvp_rel_del_r2 ~ rvat_significant', data=plot_df).fit()

(
    ggplot(plot_df, aes(y='funcrvp_rel_del_r2', x='rvat_significant')) +
    geom_point() +
    stat_smooth(method='lm') +
    annotate("text", x=40, y=max(plot_df['funcrvp_rel_del_r2']), 
        label=f"Slope: {model.params['rvat_significant']:.4f}\nP-value: {model.pvalues['rvat_significant']:.4e}", 
        ha='center', va='top', size=12, color='red') +
    xlab('RVAT discoveries') +
    ylab('FuncRVP relative delta R^2') +
    theme_bw() +
    theme(
        text=element_text(size=12)
    )
)

In [ ]:
# Define the number of bins
num_bins = 5
# Create bins for the 'signif_frac' column
plot_df['rvat_significant_bin'] = pd.qcut(plot_df['rvat_significant'], q=num_bins)

# Display the dataframe with the new binned column
plot_df

In [ ]:
# Store results for annotation
annotation_results = []

# Iterate through each bin, fit a linear model, and store the results
for name, group in plot_df.groupby('rvat_significant_bin', observed=True):
    # Fit a linear model for the current bin
    model = ols('funcrvp_rel_del_r2 ~ sampling', data=group).fit()
    # print(model.summary()) # Optional: print the full summary
    
    # Extract slope and p-value for the 'sampling' coefficient
    slope = model.params.get('sampling', float('nan')) # Use .get for safety if model fails or sampling is constant
    p_value = model.pvalues.get('sampling', float('nan'))
    
    # Store results including the bin name and max y value for positioning
    annotation_results.append({
        'rvat_significant_bin': name,
        'slope': slope,
        'p_value': p_value,
        'x_pos': group['sampling'].max(),
        'y_pos': group['funcrvp_rel_del_r2'].max() # Position annotation at the top of the y-range for the group
    })

# Convert results to DataFrame
annotation_df = pd.DataFrame(annotation_results)

# Create the label string, handling potential NaN values
annotation_df['label'] = [
    f"Slope: {s:.4f}\nP: {p:.4e}" if pd.notna(s) and pd.notna(p) else "N/A" 
    for s, p in zip(annotation_df['slope'], annotation_df['p_value'])
]

# Generate the plot with annotations
(
    ggplot(plot_df, aes(y='funcrvp_rel_del_r2', x='sampling')) +
    geom_point(alpha=0.5) + # Added alpha for potentially overlapping points
    stat_smooth(method='lm') +
    # Add text annotations using the prepared dataframe
    geom_text(
        data=annotation_df, 
        mapping=aes(x='x_pos', y='y_pos', label='label'), 
        inherit_aes=False, # Do not inherit aesthetics from the main ggplot call
        size=10,            # Adjust text size as needed
        va='top',          # Align text vertically to the top
        ha='right',       # Align text horizontally to the center
        nudge_y=-0.01 * (plot_df['funcrvp_rel_del_r2'].max() - plot_df['funcrvp_rel_del_r2'].min()) # Nudge text down slightly from max y
    ) +
    facet_wrap('rvat_significant_bin', scales='free', ncol=num_bins) + # Use free_y scales, ensure ncol matches num_bins
    theme_bw() +
    xlab('Proportion of individuals in training+validation set') +
    ylab('FuncRVP Relative Delta R^2') +
    theme(
        figure_size=(18, 6),
        text=element_text(size=16),
    )
)

In [ ]:
# Store results for annotation
annotation_results = []

# Iterate through each bin, fit a linear model, and store the results
for name, group in plot_df.groupby('sampling', observed=True):
    # Fit a linear model for the current bin
    model = ols('funcrvp_rel_del_r2 ~ rvat_significant', data=group).fit()
    # print(model.summary()) # Optional: print the full summary
    
    # Extract slope and p-value for the 'sampling' coefficient
    slope = model.params.get('rvat_significant', float('nan')) # Use .get for safety if model fails or sampling is constant
    p_value = model.pvalues.get('rvat_significant', float('nan'))
    
    # Store results including the bin name and max y value for positioning
    annotation_results.append({
        'sampling': name,
        'slope': slope,
        'p_value': p_value,
        'x_pos': group['rvat_significant'].max(),
        'y_pos': group['funcrvp_rel_del_r2'].max() # Position annotation at the top of the y-range for the group
    })

# Convert results to DataFrame
annotation_df = pd.DataFrame(annotation_results)

# Create the label string, handling potential NaN values
annotation_df['label'] = [
    f"Slope: {s:.4f}\nP: {p:.4e}" if pd.notna(s) and pd.notna(p) else "N/A" 
    for s, p in zip(annotation_df['slope'], annotation_df['p_value'])
]


# Generate the plot with annotations
(
    ggplot(plot_df, aes(y='funcrvp_rel_del_r2', x='rvat_significant')) +
    geom_point(alpha=0.5) + # Added alpha for potentially overlapping points
    stat_smooth(method='lm') +
    # Add text annotations using the prepared dataframe
    geom_text(
        data=annotation_df, 
        mapping=aes(x='x_pos', y='y_pos', label='label'), 
        inherit_aes=False, # Do not inherit aesthetics from the main ggplot call
        size=10,            # Adjust text size as needed
        va='top',          # Align text vertically to the top
        ha='right',       # Align text horizontally to the center
        nudge_y=-0.001 * (plot_df['funcrvp_rel_del_r2'].max() - plot_df['funcrvp_rel_del_r2'].min()) # Nudge text down slightly from max y
    ) +
    facet_wrap('sampling', scales='free', ncol=num_bins) + # Use free_y scales, ensure ncol matches num_bins
    theme_bw() +
    xlab('RVAT discoveries') +
    ylab('FuncRVP Relative Delta R^2') +
    theme(
        figure_size=(18, 10),
        text=element_text(size=15),
        # axis_text_x=element_text(angle=45, hjust=1), # Improve x-axis label readability if needed
    )
)

In [ ]:
from statsmodels.formula.api import ols

# Fit a linear model
model = ols('funcrvp_rel_del_r2 ~ signif_frac', data=plot_df).fit()

(
    ggplot(plot_df, aes(y='funcrvp_rel_del_r2', x='signif_frac')) +
    geom_point() +
    stat_smooth(method='lm') +
    annotate("text", x=0.5, y=max(plot_df['funcrvp_rel_del_r2']), 
        label=f"Slope: {model.params['signif_frac']:.4f}\nP-value: {model.pvalues['signif_frac']:.4e}", 
        ha='center', va='top', size=10, color='red') +
    xlab('fraction of RVAT discoveries') +
    theme_bw()
)

## Extra plots

In [ ]:
from statsmodels.formula.api import ols

# Fit a linear model
model = ols('funcrvp_lm_diff ~ sampling', data=plot_df).fit()

(
    ggplot(plot_df.query("sampling > 0.1"), aes(y='funcrvp_lm_diff', x='sampling', color='rvat_significant')) +
    geom_point() +
    stat_smooth() +
    annotate("text", x=0.5, y=max(plot_df['funcrvp_lm_diff']), 
        label=f"Slope: {model.params['sampling']:.4f}\nP-value: {model.pvalues['sampling']:.4e}", 
        ha='center', va='top', size=10, color='red') +
    theme_bw()
)